# Best DQN Showcase

Este notebook serve apenas para exibir o melhor `DQN` treinado. Ele carrega por defeito o melhor checkpoint contra `strong heuristic`, resume a run, destaca a diferença entre o pico do treino e o estado final do `self-play`, mostra curvas, corre uma avaliação mais estável e no fim permite jogar humano contra o modelo.


## Passo 1: Setup

A célula seguinte encontra a raiz do projeto, carrega os imports necessários e prepara o device.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / 'config.yaml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / 'config.yaml').exists():
    raise FileNotFoundError('Could not find project root containing config.yaml')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUTPUTS = ROOT / 'outputs'
PACKAGED_DQN_MODELS = ROOT / 'notebooks' / 'dqn' / 'outputs' / 'models' / 'dqn'

from connect4_rl.agents.baselines import HeuristicAgent, RandomAgent, StrongHeuristicAgent, WeakHeuristicAgent
from connect4_rl.config import load_config
from connect4_rl.envs.connect_four import apply_action, initial_state, is_terminal, legal_actions, outcome_for_player, render_ascii
from connect4_rl.experiments import build_agent_from_checkpoint, evaluate_against_agent, find_best_run

CONFIG = load_config(ROOT / 'config.yaml')
NOTEBOOK_DEVICE = CONFIG.resolve_device()
print({
    'torch_version': torch.__version__,
    'device': NOTEBOOK_DEVICE,
    'outputs': str(OUTPUTS),
    'packaged_models': str(PACKAGED_DQN_MODELS),
})


## Passo 2: Escolher a run

Podes apontar explicitamente para uma run, ou deixar o notebook escolher a melhor run DQN disponível em `outputs/`.


In [ ]:
run_name: str | None = 'dqn_tutorial_full_strong_focus_seed_42'

compact_candidates = [
    PACKAGED_DQN_MODELS / 'lesson3_trained_agent.pt',
    PACKAGED_DQN_MODELS / 'lesson4_trained_agent.pt',
    PACKAGED_DQN_MODELS / 'lesson2_trained_agent.pt',
    PACKAGED_DQN_MODELS / 'lesson1_trained_agent.pt',
]

metrics_data: dict[str, object]
source_mode = ''
metrics_path = None
focus_checkpoint_path = None
checkpoint_config = dict(CONFIG.dqn.__dict__)

if run_name is not None:
    candidate_metrics = OUTPUTS / run_name / 'metrics_final.json'
    if candidate_metrics.exists():
        metrics_path = candidate_metrics

if metrics_path is None:
    best_run = find_best_run(OUTPUTS, 'dqn')
    if best_run is not None:
        metrics_path = best_run.metrics_path

if metrics_path is not None:
    metrics_data = json.loads(metrics_path.read_text(encoding='utf-8'))
    focus_checkpoint_path = metrics_data.get('best_vs_strong_checkpoint_path') or metrics_data.get('best_checkpoint_path')
    checkpoint_config = dict(metrics_data.get('config', {})) or checkpoint_config
    source_mode = 'full_run_outputs'
else:
    packaged_checkpoint = next((path for path in compact_candidates if path.exists()), None)
    if packaged_checkpoint is None:
        raise RuntimeError('No DQN run found under outputs/ and no packaged checkpoint found under notebooks/dqn/outputs/models/dqn/.')
    focus_checkpoint_path = str(packaged_checkpoint)
    metrics_data = {
        'config': checkpoint_config,
        'best_checkpoint_path': str(packaged_checkpoint),
        'best_vs_strong_checkpoint_path': str(packaged_checkpoint),
        'best_score': None,
        'best_vs_strong_win_rate': None,
        'lesson_summaries': [],
        'evaluation': [],
        'phase_summary': [],
        'checkpoint_source': 'packaged_lesson_checkpoint',
    }
    source_mode = 'packaged_checkpoint'

if not focus_checkpoint_path:
    raise RuntimeError('Could not determine a DQN checkpoint to showcase.')

agent = build_agent_from_checkpoint(
    'dqn',
    focus_checkpoint_path,
    checkpoint_config,
    device=NOTEBOOK_DEVICE,
)

print({
    'source_mode': source_mode,
    'metrics_path': str(metrics_path) if metrics_path is not None else None,
    'focus_checkpoint_path': focus_checkpoint_path,
    'best_score': metrics_data.get('best_score'),
    'best_vs_strong_win_rate': metrics_data.get('best_vs_strong_win_rate'),
})


## Passo 3: Resumo da run

Mostramos o resumo da pipeline e destacamos o melhor checkpoint contra `strong heuristic`.


In [ ]:
lesson_summaries = metrics_data.get('lesson_summaries', [])
evaluation = metrics_data.get('evaluation', [])

best_vs_strong_eval = None
if evaluation:
    best_vs_strong_eval = max(
        evaluation,
        key=lambda item: float(item.get('vs_strong_heuristic_win_rate', float('-inf'))),
    )

last_eval = evaluation[-1] if evaluation else {}

print('Run summary')
print({
    'lessons': [item['lesson_name'] for item in lesson_summaries],
    'num_evaluations': len(evaluation),
    'best_checkpoint_path': metrics_data.get('best_checkpoint_path'),
    'best_vs_strong_checkpoint_path': metrics_data.get('best_vs_strong_checkpoint_path'),
    'best_vs_strong_win_rate': metrics_data.get('best_vs_strong_win_rate'),
    'checkpoint_source': metrics_data.get('checkpoint_source', 'full_run_outputs'),
})

print('\nLast evaluation point')
print(last_eval)

print('\nBest vs strong evaluation point')
print(best_vs_strong_eval)

print('\nLesson summaries')
lesson_summaries


## Passo 4: Curvas de avaliação

As curvas ajudam a perceber em que fase o modelo atingiu o melhor ponto.


In [ ]:
phase_summary = metrics_data.get('phase_summary', [])

if evaluation:
    eval_episodes = [int(item['episode']) for item in evaluation]
    vs_random = [float(item.get('vs_random_win_rate', 0.0)) for item in evaluation]
    vs_weak = [float(item.get('vs_weak_heuristic_win_rate', 0.0)) for item in evaluation]
    vs_strong = [float(item.get('vs_strong_heuristic_win_rate', 0.0)) for item in evaluation]
    eval_outcome = [float(item.get('eval_mean_outcome', 0.0)) for item in evaluation]
    best_vs_strong_idx = max(range(len(vs_strong)), key=lambda idx: vs_strong[idx])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(eval_episodes, vs_random, marker='o', label='vs random')
    axes[0].plot(eval_episodes, vs_weak, marker='o', label='vs weak')
    axes[0].plot(eval_episodes, vs_strong, marker='o', label='vs strong')
    axes[0].scatter([eval_episodes[best_vs_strong_idx]], [vs_strong[best_vs_strong_idx]], color='red', s=80, label='best vs strong')
    axes[0].set_ylim(0.0, 1.0)
    axes[0].set_title('Win rates at evaluation checkpoints')
    axes[0].set_xlabel('Episode')
    axes[0].grid(alpha=0.2)
    axes[0].legend()

    axes[1].plot(eval_episodes, eval_outcome, marker='o', color='black')
    axes[1].axhline(0.0, color='gray', linestyle='--', alpha=0.4)
    axes[1].set_title('Eval mean outcome')
    axes[1].set_xlabel('Episode')
    axes[1].grid(alpha=0.2)

    for ax in axes:
        for phase in phase_summary:
            ax.axvline(int(phase['end_episode']), color='gray', linestyle='--', alpha=0.2)

    plt.tight_layout()
    plt.show()
else:
    print('No evaluation history was found. The notebook is using a packaged checkpoint only.')


## Passo 5: Avaliação final mais estável

Corremos mais jogos do que no loop de treino para ter uma leitura melhor do modelo.


In [5]:
def evaluate_suite(model, games: int = 80) -> dict[str, float]:
    return {
        'vs_random': evaluate_against_agent(model, lambda idx: RandomAgent(seed=10_000 + idx), games=games),
        'vs_weak': evaluate_against_agent(model, lambda idx: WeakHeuristicAgent(seed=20_000 + idx), games=games),
        'vs_strong': evaluate_against_agent(model, lambda idx: StrongHeuristicAgent(seed=30_000 + idx), games=games),
        'vs_heuristic': evaluate_against_agent(model, lambda idx: HeuristicAgent(seed=40_000 + idx), games=games),
    }

showcase_eval = evaluate_suite(agent, games=80)
showcase_eval


{'vs_random': 0.85,
 'vs_weak': 0.925,
 'vs_strong': 0.8875,
 'vs_heuristic': 0.8125}

## Passo 6: Partidas exemplo

Estas partidas em ASCII ajudam a perceber o comportamento do modelo sem depender de gráficos.


In [6]:
def play_and_render(model, opponent, controlled_player: int = 1) -> str:
    state = initial_state()
    transcript = ['Initial board', render_ascii(state), '']
    move_idx = 0
    while not is_terminal(state):
        move_idx += 1
        if state.current_player == controlled_player:
            action = model.select_action(state, legal_actions(state))
            actor = model.name
        else:
            action = opponent.select_action(state, legal_actions(state))
            actor = opponent.name
        state = apply_action(state, action)
        transcript.append(f'Move {move_idx}: {actor} played column {action}')
        transcript.append(render_ascii(state))
        transcript.append('')
    transcript.append(f'Winner: {state.winner}')
    return '\n'.join(transcript)

print(play_and_render(agent, RandomAgent(seed=1), controlled_player=1))


Initial board
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
0 1 2 3 4 5 6

Move 1: dqn played column 3
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . X . . .
0 1 2 3 4 5 6

Move 2: random played column 1
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. O . X . . .
0 1 2 3 4 5 6

Move 3: dqn played column 3
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . X . . .
. O . X . . .
0 1 2 3 4 5 6

Move 4: random played column 4
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . X . . .
. O . X O . .
0 1 2 3 4 5 6

Move 5: dqn played column 4
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . X X . .
. O . X O . .
0 1 2 3 4 5 6

Move 6: random played column 6
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . X X . .
. O . X O . O
0 1 2 3 4 5 6

Move 7: dqn played column 4
. . . . . . .
. . . . . . .
. . . . . . .
. . . . X . .
. . . X X . .
. O . X O . O
0 1 

In [7]:
print(play_and_render(agent, HeuristicAgent(seed=2), controlled_player=1))


Initial board
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
0 1 2 3 4 5 6

Move 1: dqn played column 3
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . X . . .
0 1 2 3 4 5 6

Move 2: heuristic played column 6
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . X . . O
0 1 2 3 4 5 6

Move 3: dqn played column 3
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . X . . .
. . . X . . O
0 1 2 3 4 5 6

Move 4: heuristic played column 5
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . X . . .
. . . X . O O
0 1 2 3 4 5 6

Move 5: dqn played column 2
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . X . . .
. . X X . O O
0 1 2 3 4 5 6

Move 6: heuristic played column 4
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . X . . .
. . X X O O O
0 1 2 3 4 5 6

Move 7: dqn played column 4
. . . . . . .
. . . . . . .
. . . . . . .
. . . . . . .
. . . X X . .
. . X X O

## Passo 7: Humano vs modelo

A célula seguinte usa `input()` e foi pensada para correr num kernel interativo. Escolhe `human_player = 1` para jogar primeiro, ou `2` para jogar em segundo.


In [8]:
def play_human_vs_model(model, human_player: int = 1) -> None:
    if human_player not in (1, 2):
        raise ValueError('human_player must be 1 or 2')

    state = initial_state()
    print('Starting game')
    print(render_ascii(state))

    while not is_terminal(state):
        legal = legal_actions(state)
        print(f'Current player: {state.current_player} | Legal actions: {legal}')

        if state.current_player == human_player:
            while True:
                raw = input('Choose a column: ').strip()
                try:
                    action = int(raw)
                except ValueError:
                    print('Please enter an integer column index.')
                    continue
                if action not in legal:
                    print(f'Illegal move. Choose one of {legal}.')
                    continue
                break
            actor = 'human'
        else:
            action = model.select_action(state, legal)
            actor = model.name

        state = apply_action(state, action)
        print(f'\n{actor} played column {action}')
        print(render_ascii(state))
        print()

    result = outcome_for_player(state, human_player)
    if result > 0:
        print('Human wins.')
    elif result < 0:
        print('Model wins.')
    else:
        print('Draw.')

# Example:
# play_human_vs_model(agent, human_player=1)


In [ ]:
# Example:
# play_human_vs_model(agent, human_player=1)
